In [1]:
import numpy as np

## Neural Network — 4-layer MLP with full backprop

**The problem.** Logistic regression could only draw a *straight* boundary. Real data is often tangled in ways no line can separate. A neural network stacks several linear layers with **non-linear** activations in between, and that stack can bend itself into arbitrarily complex shapes.

**The architecture.** A multilayer perceptron (MLP) with **4 weight layers**, sizes `[d → 64 → 32 → 16 → C]`. Each hidden layer applies a linear transform then a **ReLU** ($\max(0, z)$, which simply zeroes out negatives and lets positives through). The final layer applies **softmax**, turning raw scores into a probability distribution over the $C$ classes.

**The forward pass.** For layer $\ell$, compute a linear combination of the previous layer's outputs, then apply the activation:
$$z^{(\ell)} = a^{(\ell-1)} W^{(\ell)} + b^{(\ell)}, \qquad a^{(\ell)} = \text{ReLU}(z^{(\ell)})$$
with softmax replacing ReLU on the last layer. We *cache* the intermediate $z$ and $a$ values because the backward pass needs them.

**The backward pass (backpropagation).** Training means computing $\partial L/\partial W^{(\ell)}$ for every layer. Backprop does this efficiently by carrying an error signal $\delta$ from the output back toward the input, reusing work via the chain rule. Two facts make it clean:

- At the output, softmax + cross-entropy collapse to the same tidy form we keep seeing: $\;\delta^{(L)} = \hat{y} - y$.
- To move $\delta$ one layer earlier, multiply by that layer's weights and then mask by the ReLU derivative, which is just an on/off gate: $\text{ReLU}'(z) = \mathbb{1}[z>0]$.

The per-layer gradients are then:
$$\frac{\partial L}{\partial W^{(\ell)}} = a^{(\ell-1)\top}\delta^{(\ell)}, \qquad
\frac{\partial L}{\partial b^{(\ell)}} = \sum_i \delta^{(\ell)}_i, \qquad
\delta^{(\ell-1)} = \big(\delta^{(\ell)} W^{(\ell)\top}\big)\odot \text{ReLU}'(z^{(\ell-1)})$$
($\odot$ is elementwise multiplication.) The code below implements exactly these three lines in a loop that walks the layers in reverse.

In [96]:
import numpy as np


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


class NeuralNetwork():

    def __init__(self, sizes, lr=0.1, epochs = 100):
        self.L = len(sizes) - 1
        self.lr = lr
        self.epochs = epochs
        self.W, self.b = [], []

        for i in range(1, len(sizes)):
            # He init: scale by sqrt(2 / fan_in), where fan_in = sizes[i-1]
            self.W.append(np.random.randn(sizes[i], sizes[i-1]) * np.sqrt(2 / sizes[i-1]))
            self.b.append(np.zeros((sizes[i], 1)))

    def forward(self, X):
        # coerce input to a column vector: shape (n_features, 1)
        h = np.asarray(X, dtype=float).reshape(-1, 1)
        # print(f"h = {h}")
        self.z, self.a = [], [h]

        for i in range(self.L):
            z = self.W[i] @ h + self.b[i]
            self.z.append(z)
            h = sigmoid(z) if i == self.L - 1 else np.maximum(0, z)
            self.a.append(h)

        return h

    def backward(self, y):
        y = np.asarray(y, dtype=float).reshape(-1, 1)
        # print(f"y = {y}")
        delta = self.a[-1] - y   # assumes sigmoid output + BCE loss

        for i in reversed(range(self.L)):
            grad_W = delta @ self.a[i].T          # (out,1) @ (1,in) -> (out,in)
            grad_b = delta                        # (out,1)

            if i > 0:
                delta = (self.W[i].T @ delta) * (self.z[i - 1] > 0)

            self.W[i] -= self.lr * grad_W
            self.b[i] -= self.lr * grad_b

    def loss(self, X, y):
        eps = 1e-9
        total = 0.0
        for xi, yi in zip(X, y):
            p = self.forward(xi).ravel()
            total += -(yi * np.log(p + eps) + (1 - yi) * np.log(1 - p + eps)).sum()
        return total / len(y)


    def fit(self, X, y, epochs=1000, verbose=True):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        n = len(X)
        for epoch in range(self.epochs):
            for idx in np.random.permutation(n):   # shuffle each epoch
                self.forward(X[idx])
                self.backward(y[idx])
            if verbose and (epoch % 100 == 0 or epoch == epochs - 1):
                print(f"epoch {epoch:5d}  loss {self.loss(X, y):.4f}")
        return self


    def predict(self, X):
        return np.array([self.forward(xi).ravel() for xi in X])


NN = NeuralNetwork([2, 4, 3, 1])
out = NN.forward([1, 1])
print("output:", out.ravel())
NN.backward(np.array([0]))

output: [0.65693122]


In [97]:
# XOR
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([0, 1, 1, 0])

NN = NeuralNetwork([2, 8, 1], lr=0.1, epochs=2000)
NN.fit(X, y)
print("preds:", np.round(NN.predict(X), 3))

epoch     0  loss 0.6509
epoch   100  loss 0.0708
epoch   200  loss 0.0284
epoch   300  loss 0.0171
epoch   400  loss 0.0121
epoch   500  loss 0.0093
epoch   600  loss 0.0075
epoch   700  loss 0.0063
epoch   800  loss 0.0054
epoch   900  loss 0.0047
epoch   999  loss 0.0042
epoch  1000  loss 0.0042
epoch  1100  loss 0.0038
epoch  1200  loss 0.0034
epoch  1300  loss 0.0031
epoch  1400  loss 0.0029
epoch  1500  loss 0.0027
epoch  1600  loss 0.0025
epoch  1700  loss 0.0023
epoch  1800  loss 0.0022
epoch  1900  loss 0.0021
preds: [[0.003]
 [0.999]
 [0.999]
 [0.003]]
